# SAR Data Acquisition and Pixel Offset Tracking

RTC example
https://github.com/ASFHyP3/hyp3-docs/blob/main/docs/tutorials/new-rtc-jobs.ipynb


Insar, rtc, autorift examples
https://github.com/ASFHyP3/hyp3-sdk/blob/main/docs/sdk_example.ipynb 

RTC prouct guide 
https://hyp3-docs.asf.alaska.edu/guides/rtc_product_guide/#incidence-angle-map

In [11]:
import asf_search
import hyp3_sdk as sdk

from pyproj import Proj, Geod
P = Proj('epsg:32637')
G = Geod(ellps='WGS84')

In [12]:
username = 'chanagan@usgs.gov'
password = 'LPycN-i%8H3%VzE'

hyp3 = sdk.HyP3(username=username, password=password)

In [13]:
project_name = 'TurkeyTest'

In [ ]:
start = "2023-01-27T00:00:00Z"
end = "2023-02-10T00:00:00Z"

# Bounds (te) minx, miny, maxx, maxy
tep = [325910.529, 4198948.4196, 355327.725, 4221973.271]
te = [
    P(tep[0], tep[1], inverse=True)[0],  # lon_min
    P(tep[0], tep[1], inverse=True)[1],  # lat_min
    P(tep[2], tep[3], inverse=True)[0],  # lon_max
    P(tep[2], tep[3], inverse=True)[1],  # lat_max
]

poly = f"POLYGON(({te[0]} {te[3]}, {te[1]} {te[3]}, {te[1]} {te[2]}, {te[0]} {te[1]}, {te[0]} {te[3]}))"
poly

'POLYGON((37.019352464912636 38.134030419594, 37.921440237693886 38.134030419594, 37.921440237693886 37.3492475901632, 37.019352464912636 37.921440237693886, 37.019352464912636 38.134030419594))'

# RTC

In [19]:
search_parameters = {
    "start": start,
    "end": end,
    "intersectsWith": poly,
    "platform": "S1",
    "processingLevel": "SLC",
}
job_specification = {
    "job_parameters": {
        "resolution": 10, # 10, 20, 30 m, 
        "include_inc_map": True,
    },
    "job_type": "RTC_GAMMA",
    "name": project_name
}

In [25]:
previous_jobs = hyp3.find_jobs(
    name=job_specification['name'],
    job_type=job_specification['job_type'],
)
processed_granules = [job.job_parameters['granules'][0] for job in previous_jobs]
print(f'Found {len(processed_granules)} previously processed granules')

search_results = asf_search.search(**search_parameters)
search_results.raise_if_incomplete()

unprocessed_granules = [
    result for result in search_results if result.properties['sceneName'] not in processed_granules
]
print(f'Found {len(unprocessed_granules)} unprocessed granules')

/var/folders/zn/w54gt7k11csfs1n4z8r3xccsyqhmjz/T/ipykernel_6979/614427691.py:8: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  search_results = asf_search.search(**search_parameters)
["'type': 'REVERSE': 'report': Reversed polygon winding order"]


Found 5 previously processed granules


/var/folders/zn/w54gt7k11csfs1n4z8r3xccsyqhmjz/T/ipykernel_6979/614427691.py:8: DeprecationWarning: Parsing dates involving a day of month without a year specified is ambiguous
and fails to parse leap day. The default behavior will change in Python 3.15
to either always raise an exception or to use a different default year (TBD).
To avoid trouble, add a specific year to the input & format.
See https://github.com/python/cpython/issues/70647.
  search_results = asf_search.search(**search_parameters)


Found 0 unprocessed granules


In [53]:
unprocessed_granules = [search_results[-1]]

In [54]:
from copy import deepcopy

def get_job_for_granule(granule: asf_search.ASFProduct) -> dict:
    job = deepcopy(job_specification)
    job['job_parameters']['granules'] = [granule.properties['sceneName']]
    return job


batch_size = 1
for i in range(0, len(unprocessed_granules), batch_size):
    new_jobs = [
        get_job_for_granule(granule) for granule in unprocessed_granules[i:i+batch_size]
    ]
    print(f'Submitting {len(new_jobs)} jobs')
    hyp3.submit_prepared_jobs(new_jobs)

print('Done.')

Submitting 1 jobs
Done.


In [56]:
rtc_jobs = hyp3.find_jobs(name=project_name)
rtc_jobs = hyp3.watch(rtc_jobs)

100%|██████████| 8/8 [timeout in 7500s]  


In [58]:
print(f'Total number of Jobs: {len(rtc_jobs)}')
succeeded_jobs = rtc_jobs.filter_jobs(succeeded=True, running=False, failed=False)
print(f'Number of succeeded jobs: {len(succeeded_jobs)}')
failed_jobs = rtc_jobs.filter_jobs(succeeded=False, running=False, failed=True)
print(f'Number of failed jobs: {len(failed_jobs)}')

Total number of Jobs: 8
Number of succeeded jobs: 8
Number of failed jobs: 0


In [60]:
for i in succeeded_jobs:
    print(i.job_parameters, i.files[0]['url'])

{'speckle_filter': False, 'include_inc_map': True, 'dem_name': 'copernicus', 'radiometry': 'gamma0', 'granules': ['S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8'], 'scale': 'power', 'dem_matching': False, 'resolution': 10, 'include_rgb': False, 'include_dem': False, 'include_scattering_area': False} https://d3gm2hf49xd6jj.cloudfront.net/29fffcb0-3572-407b-b542-cabcc4b73a7b/S1A_IW_20230128T153424_DVP_RTC10_G_gpuned_2155.zip
{'speckle_filter': False, 'include_inc_map': True, 'dem_name': 'copernicus', 'radiometry': 'gamma0', 'granules': ['S1A_IW_SLC__1SDV_20230205T032616_20230205T032644_047095_05A659_8912'], 'scale': 'power', 'dem_matching': False, 'resolution': 10, 'include_rgb': False, 'include_dem': False, 'include_scattering_area': False} https://d3gm2hf49xd6jj.cloudfront.net/348eec9b-03d2-405e-a6a4-9396c654e735/S1A_IW_20230205T032616_DVP_RTC10_G_gpuned_C311.zip
{'speckle_filter': False, 'include_inc_map': True, 'dem_name': 'copernicus', 'radiometry': 'gamma0', '

In [64]:
stop%%bash
gdalwarp -t_srs EPSG:32637 -overwrite -crop_to_cutline -cutline /Users/chanagan/Library/CloudStorage/OneDrive-DOI/ToShare/DeformationSynthesisSharedResources/GIS/AOIepsg32637.geojson -tr 10 -10 -r near -of GTiff -ot Float32 -dstnodata -9999 \
        /Users/chanagan/Downloads/TurkeyTest/S1A_IW_20230128T153424_DVP_RTC10_G_gpuned_2155/S1A_IW_20230128T153424_DVP_RTC10_G_gpuned_2155_VV.tif \
            /Users/chanagan/Downloads/TurkeyTest/S1A_IW_20230128T153424_DVP_RTC10_G_gpuned_2155/S1A_IW_20230128T153424_DVP_RTC10_G_gpuned_2155_VV_toAOI.tif

gdalwarp -t_srs EPSG:32637 -overwrite -crop_to_cutline -cutline /Users/chanagan/Library/CloudStorage/OneDrive-DOI/ToShare/DeformationSynthesisSharedResources/GIS/AOIepsg32637.geojson -tr 10 -10 -r near -of GTiff -ot Float32 -dstnodata -9999 \
        /Users/chanagan/Downloads/TurkeyTest/S1A_IW_20230209T153423_DVP_RTC10_G_gpuned_3695/S1A_IW_20230209T153423_DVP_RTC10_G_gpuned_3695_VV.tif \ 
        /Users/chanagan/Downloads/TurkeyTest/S1A_IW_20230209T153423_DVP_RTC10_G_gpuned_3695/S1A_IW_20230209T153423_DVP_RTC10_G_gpuned_3695_VV_toAOI.tif

SyntaxError: invalid syntax (1939830669.py, line 1)

In [ ]:
mm3d Mm2dPosSism S1A_IW_20230128T153424_DVP_RTC10_G_gpuned_2155_VV_toAOI.tif S1A_IW_20230209T153423_DVP_RTC10_G_gpuned_3695_VV_toAOI.tif CorMin=0.1 Dequant=false DirMEC='MEC_RTC10m/'  

In [1]:
import sys
sys.path.append('/Users/chanagan/Documents/GitHub/ImageryResources/Functions/')
import TiffTools as tt

# reload modules when changed
%reload_ext autoreload
%autoreload 2

In [2]:
tt.micmacPostProcessing(folder=f'/Users/chanagan/Downloads/TurkeyTest/MEC_RTC10m/',
                             prefiles=[f'/Users/chanagan/Downloads/TurkeyTest/S1A_IW_20230128T153424_DVP_RTC10_G_gpuned_2155_VV_toAOI.tif',
                                       f'/Users/chanagan/Downloads/TurkeyTest/S1A_IW_20230209T153423_DVP_RTC10_G_gpuned_3695_VV_toAOI.tif'],
                             outprefix=f'/Users/chanagan/Downloads/TurkeyTest/MEC_RTC10m/')

/Users/chanagan/miniforge3/envs/image_processing/lib/python3.13/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Nodata value for mask: -9999.0
Setting nodata value to -9999
Saving /Users/chanagan/Downloads/TurkeyTest/MEC_RTC10m/NSmicmac.tif
Saving /Users/chanagan/Downloads/TurkeyTest/MEC_RTC10m/EWmicmac.tif
Saving /Users/chanagan/Downloads/TurkeyTest/MEC_RTC10m/Correlmicmac.tif


# SLC search, download, clip for autorift

In [ ]:
# SLC Search

In [57]:
# DEM from open topo
!gdalwarp -t_srs EPSG:32637 -overwrite -crop_to_cutline -cutline /Users/chanagan/Library/CloudStorage/OneDrive-DOI/ToShare/DeformationSynthesisSharedResources/GIS/AOIepsg32637.geojson -tr 10 -10 -r cubicspline -of GTiff -ot Float32 -dstnodata -9999 \
    /Users/chanagan/Downloads/TurkeyTest/cop30.tif \
        /Users/chanagan/Downloads/TurkeyTest/cop30toAOI.tif 


Creating output file that is 2942P x 2302L.
Processing cop30.tif [1/1] : 0...10...20...30...40...50...60...70...80...90...100 - done.


In [ ]:
!./Users/chanagan/Documents/GitHub/autoRIFT/testautoRIFT.py -m /Users/chanagan/Downloads/TurkeyTest/S1A_IW_SLC__1SDV_20230209T153423_20230209T153450_047161_05A88A_9FEC.SAFE/measurement/s1a-iw2-slc-vv-20230209t153425-20230209t153450-047161-05a88a-005.tiff \
    -s /Users/chanagan/Downloads/TurkeyTest/S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8.SAFE/measurement/s1a-iw2-slc-vv-20230128t153426-20230128t153451-046986-05a2b6-005.tiff \


In [ ]:
!./Users/chanagan/Documents/GitHub/autoRIFT/testGeogrid.py -m /Users/chanagan/Downloads/TurkeyTest/S1A_IW_SLC__1SDV_20230209T153423_20230209T153450_047161_05A88A_9FEC.SAFE/measurement/s1a-iw2-slc-vv-20230209t153425-20230209t153450-047161-05a88a-005.tiff \
    -s /Users/chanagan/Downloads/TurkeyTest/S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8.SAFE/measurement/s1a-iw2-slc-vv-20230128t153426-20230128t153451-046986-05a2b6-005.tiff \
    -d /Users/chanagan/Downloads/TurkeyTest/cop30toAOI.tif

# Autorift

https://github.com/nasa-jpl/autoRIFT/blob/main/docs/demo.md

In [ ]:
!/Users/chanagan/Documents/GitHub/autoRIFT/testGeogrid.py 

# Autorift - hyp3 (only for ice)




In [ ]:
autorift_pairs = [('S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8',
                'S1A_IW_SLC__1SDV_20230209T153423_20230209T153450_047161_05A88A_9FEC')]

In [58]:
autorift_jobs = sdk.Batch()
for reference, secondary in autorift_pairs:
    autorift_jobs += hyp3.submit_autorift_job(reference, secondary, name=project_name+'autorift')
print(autorift_jobs)

1 HyP3 Jobs: 0 succeeded, 0 failed, 0 running, 1 pending.


In [61]:
hyp3.find_jobs(name=project_name+'autorift')

Batch([Job.from_dict({'credit_cost': 25, 'priority': 7925, 'job_parameters': {'granules': ['S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8', 'S1A_IW_SLC__1SDV_20230221T153423_20230221T153450_047336_05AE7F_A71C']}, 'job_id': 'a8fe065b-144a-40ef-9c68-31a43b0992ea', 'name': 'TurkeyTestautorift', 'user_id': 'cehanagan', 'status_code': 'RUNNING', 'request_time': '2026-01-26T18:55:18+00:00', 'job_type': 'AUTORIFT'})])

In [7]:
autorift_jobs = hyp3.find_jobs(name=project_name+'autorift')
autorift_jobs = hyp3.watch(autorift_jobs)

100%|██████████| 1/1 [timeout in 7560s]  


In [8]:
print(f'Total number of Jobs: {len(autorift_jobs)}')
succeeded_jobs = autorift_jobs.filter_jobs(succeeded=True, running=False, failed=False)
print(f'Number of succeeded jobs: {len(succeeded_jobs)}')
failed_jobs = autorift_jobs.filter_jobs(succeeded=False, running=False, failed=True)
print(f'Number of failed jobs: {len(failed_jobs)}')

Total number of Jobs: 1
Number of succeeded jobs: 1
Number of failed jobs: 0


In [9]:
file_list = succeeded_jobs.download_files(location='./')

S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8_X_S1A_IW_SLC__1SDV_20230221T153423_20230221T153450_047336_05AE7F_A71C_G0120V02_P093_IL_ASF_OD.nc: 100%|██████████| 437k/437k [00:00<00:00, 1.39MB/s]
100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


In [10]:
succeeded_jobs

Batch([Job.from_dict({'files': [{'s3': {'bucket': 'hyp3-edc-prod-contentbucket-1fv14ed36ifj6', 'key': 'a8fe065b-144a-40ef-9c68-31a43b0992ea/S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8_X_S1A_IW_SLC__1SDV_20230221T153423_20230221T153450_047336_05AE7F_A71C_G0120V02_P093_IL_ASF_OD.nc'}, 'filename': 'S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8_X_S1A_IW_SLC__1SDV_20230221T153423_20230221T153450_047336_05AE7F_A71C_G0120V02_P093_IL_ASF_OD.nc', 'size': 447585, 'url': 'https://d3gm2hf49xd6jj.cloudfront.net/a8fe065b-144a-40ef-9c68-31a43b0992ea/S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8_X_S1A_IW_SLC__1SDV_20230221T153423_20230221T153450_047336_05AE7F_A71C_G0120V02_P093_IL_ASF_OD.nc'}], 'status_code': 'SUCCEEDED', 'job_type': 'AUTORIFT', 'job_id': 'a8fe065b-144a-40ef-9c68-31a43b0992ea', 'priority': 7925, 'name': 'TurkeyTestautorift', 'user_id': 'cehanagan', 'expiration_time': '2026-02-10T00:00:00+00:00', 'thumbnail_images': ['

# ISCE and Autorift

In [ ]:
from pyproj import Proj
from shapely.geometry import Polygon

start = "2023-01-27T00:00:00Z"
end   = "2023-02-10T00:00:00Z"

# Projected bounds (UTM assumed)
tep = [325910.529, 4198948.4196, 355327.725, 4221973.271]

# Projection object (example: UTM Zone 11N – adjust if needed)
P = Proj("EPSG:32611")

te = [
    P(tep[0], tep[1], inverse=True)[0],  # lon_min
    P(tep[0], tep[1], inverse=True)[1],  # lat_min
    P(tep[2], tep[3], inverse=True)[0],  # lon_max
    P(tep[2], tep[3], inverse=True)[1],  # lat_max
]

poly = f"POLYGON(({te[0]} {te[3]}, {te[1]} {te[3]}, {te[1]} {te[2]}, {te[0]} {te[1]}, {te[0]} {te[3]}))"
poly


In [ ]:
import asf_search as asf

results = asf.search(
    platform=asf.PLATFORM.SENTINEL1,
    processingLevel=asf.PRODUCT_TYPE.SLC,
    start=start,
    end=end,
    intersectsWith=poly,
)

print(f"Found {len(results)} scenes")
results


In [ ]:
# Example: pick first scene, then match orbit
ref = results[0]

pair = [
    r for r in results
    if r.properties['pathNumber'] == ref.properties['pathNumber']
    and r.properties['flightDirection'] == ref.properties['flightDirection']
]

pair


In [ ]:
username = 'chanagan@usgs.gov'
password = 'LPycN-i%8H3%VzE'

session = asf.ASFSession().auth_with_creds(username, password)
asf.download(pair, path="SLCs/", session=session)


Directory structure: 

project/ \
├── SLCs/ \
│   ├── S1A_*.SAFE \
├── DEM/ \
│   └── dem.tif \
├── isce/ \
│   └── topsApp.xml 


In [3]:
<topsApp>
  <component name="topsinsar">
    <property name="Sensor name">SENTINEL1</property>
    <property name="reference directory">/Users/chanagan/Downloads/TurkeyTest/S1A_IW_SLC__1SDV_20230128T153424_20230128T153451_046986_05A2B6_00C8.SAFE</property>
    <property name="secondary directory">/Users/chanagan/Downloads/TurkeyTest/S1A_IW_SLC__1SDV_20230209T153423_20230209T153450_047161_05A88A_9FEC.SAFE</property>
    <property name="do dense offsets">False</property>
    <property name="dem filename">/Users/chanagan/Downloads/TurkeyTest/cop30toAOI.tif</property>
  </component>
</topsApp>


SyntaxError: invalid syntax (2717435064.py, line 1)

In [ ]:
import subprocess
subprocess.run(["topsApp.py", "topsApp.xml"], check=True)